# 數據並行 — 自無至有

> **難易：** 初學 → 中等 | **所需時辰：** 約四刻

此篇講述**數據並行（Data Parallelism, DP）**——分佈式深度學習之最基本、最廣用之術。

**毋需分佈式訓練之先備知識。** 吾等自根本起講。

## 一、何以需多器？

設有塾師一人，批閱試卷千份——獨力為之，曠日持久。
然若四師各閱二百五十份，末了**匯集所得**，則事半功倍，速近四倍。

**數據並行之理，與此無異：**
- 每張 GPU 猶一塾師
- 試卷即訓練數據
- 「匯集所得」即同步各 GPU 所學之梯度

先繪一圖，後詳其義。

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "../..")))

from mp_tutorial.distributed import check_gpu_env, simulate_allreduce
from mp_tutorial.viz import draw_comm_pattern
from mp_tutorial.formatting import info_box, gpu_required_banner, comparison_table, code_reference

### 要義一圖

下圖盡含數據並行之全部要義。先觀此圖，後文皆繞之而展。

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
ax.set_xlim(0, 11)
ax.set_ylim(0, 6.5)
ax.axis("off")
ax.set_title("數據並行 — 全覽圖", fontsize=15, fontweight="bold", pad=12)

gpu_colors = ["#4FC3F7", "#81C784", "#FFB74D", "#E57373"]

# 完整批次
ax.add_patch(patches.FancyBboxPatch((1.5, 5.3), 8, 0.8, boxstyle="round,pad=0.05",
    facecolor="#eeeeee", edgecolor="#333", lw=1.5))
ax.text(5.5, 5.7, "全量訓練批次（如 256 樣本）", ha="center", va="center",
    fontsize=12, fontweight="bold")

# 分割箭頭
for i in range(4):
    x = 2.0 + i * 2.0
    ax.annotate("", xy=(x, 4.9), xytext=(5.5, 5.3),
        arrowprops=dict(arrowstyle="->,head_width=0.1", color="#888", lw=1.2))

# 四器各列
for i in range(4):
    x = 1.2 + i * 2.0
    # 微批次
    ax.add_patch(patches.FancyBboxPatch((x, 4.0), 1.6, 0.7, boxstyle="round,pad=0.04",
        facecolor=gpu_colors[i], edgecolor="#333", lw=1, alpha=0.7))
    ax.text(x + 0.8, 4.35, f"64 樣本", ha="center", va="center", fontsize=9, color="white",
        fontweight="bold")

    # 模型副本
    ax.add_patch(patches.FancyBboxPatch((x, 2.6), 1.6, 1.1, boxstyle="round,pad=0.04",
        facecolor="#e3f2fd", edgecolor="#1565c0", lw=1.5))
    ax.text(x + 0.8, 3.3, "同一模型", ha="center", va="center", fontsize=10, fontweight="bold")
    ax.text(x + 0.8, 3.0, "（完整副本）", ha="center", va="center", fontsize=8, color="#666")

    ax.annotate("", xy=(x + 0.8, 3.75), xytext=(x + 0.8, 4.0),
        arrowprops=dict(arrowstyle="->,head_width=0.08", color="#555", lw=1))

    # 梯度
    ax.add_patch(patches.FancyBboxPatch((x, 1.4), 1.6, 0.7, boxstyle="round,pad=0.04",
        facecolor="#fff9c4", edgecolor="#f9a825", lw=1))
    ax.text(x + 0.8, 1.75, f"梯度 {i}", ha="center", va="center", fontsize=9, fontweight="bold")

    ax.annotate("", xy=(x + 0.8, 2.15), xytext=(x + 0.8, 2.6),
        arrowprops=dict(arrowstyle="->,head_width=0.08", color="#555", lw=1))

    ax.text(x + 0.8, 4.85, f"GPU {i}", ha="center", va="center", fontsize=10, fontweight="bold",
        color=gpu_colors[i])

# 同步
ax.add_patch(patches.FancyBboxPatch((2.5, 0.3), 6, 0.7, boxstyle="round,pad=0.05",
    facecolor="#e8f5e9", edgecolor="#2e7d32", lw=1.5))
ax.text(5.5, 0.65, "AllReduce — 匯總梯度取均值 → 各器以同一梯度更新",
    ha="center", va="center", fontsize=10, fontweight="bold", color="#1b5e20")

for i in range(4):
    x = 2.0 + i * 2.0
    ax.annotate("", xy=(x, 1.05), xytext=(x, 1.4),
        arrowprops=dict(arrowstyle="->,head_width=0.08", color="#2e7d32", lw=1.2))

plt.tight_layout()
plt.show()

### 釋名（初學者必讀）

深入之前，先明數術語之義：

In [ ]:
info_box(
    "<b>張量（Tensor）</b> — 多維數組也。一維為向量，二維為矩陣。模型權重與數據皆張量。<br><br>"
    "<b>前向傳播（Forward Pass）</b> — 以輸入數據過模型，得預測之結果。<br><br>"
    "<b>損失（Loss）</b> — 一標量數，衡預測之誤差。愈小愈佳。<br><br>"
    "<b>反向傳播（Backward Pass）</b> — 算各權重對損失之貢獻。所得即每權重之<i>梯度</i>。<br><br>"
    "<b>梯度（Gradient）</b> — 與權重同形之張量，示如何調權重以減損失。<br><br>"
    "<b>優化步（Optimizer Step）</b> — 以梯度實際更新權重"
    "（如 <code>weight -= learning_rate × gradient</code>）。",
    title="釋名 — 深度學習之基本術語"
)

## 二、單器訓練（基準）

未論並行，先觀**單卡**之訓練流程。
數據並行者，不過使多卡同時行此流程耳。

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.set_xlim(0, 10)
ax.set_ylim(0, 3)
ax.axis("off")
ax.set_title("單 GPU 訓練循環（一步）", fontsize=13, fontweight="bold", pad=8)

steps = [
    (0.3, "輸入\n數據", "#e3f2fd", "#1565c0"),
    (2.3, "前向\n傳播", "#e8f5e9", "#2e7d32"),
    (4.3, "計算\n損失", "#fff3e0", "#e65100"),
    (6.3, "反向\n傳播", "#fce4ec", "#c62828"),
    (8.3, "更新\n權重", "#f3e5f5", "#6a1b9a"),
]
for i, (x, label, fc, ec) in enumerate(steps):
    ax.add_patch(patches.FancyBboxPatch((x, 0.8), 1.5, 1.4, boxstyle="round,pad=0.06",
        facecolor=fc, edgecolor=ec, lw=1.5))
    ax.text(x + 0.75, 1.5, label, ha="center", va="center", fontsize=10, fontweight="bold")
    if i < len(steps) - 1:
        ax.annotate("", xy=(x + 1.7, 1.5), xytext=(x + 1.55, 1.5),
            arrowprops=dict(arrowstyle="->,head_width=0.1", color="#555", lw=1.5))

plt.tight_layout()
plt.show()

### 以實數觀之

於微型模型上行一步訓練，觀每數之變化：

In [ ]:
torch.manual_seed(42)

# 微型模型：單線性層（2 輸入 → 3 輸出）
model = nn.Linear(2, 3, bias=False)
print("模型權重（訓練前）：")
print(model.weight.data)
print(f"形狀：{model.weight.shape}  (3 輸出 × 2 輸入)\n")

# 小批次：4 樣本，各有 2 特徵
X = torch.tensor([[1.0, 0.0],
                   [0.0, 1.0],
                   [1.0, 1.0],
                   [0.5, 0.5]])
y_true = torch.tensor([[1.0, 0.0, 0.0],
                        [0.0, 1.0, 0.0],
                        [1.0, 1.0, 0.0],
                        [0.5, 0.5, 0.0]])

# 前向傳播
y_pred = model(X)
print("預測結果（前向傳播）：")
print(y_pred.data.round(decimals=3))

# 計算損失
loss = ((y_pred - y_true) ** 2).mean()
print(f"\n損失值：{loss.item():.4f}")

# 反向傳播
loss.backward()
print(f"\n梯度（與權重同形）：")
print(model.weight.grad.round(decimals=3))

# 更新權重
lr = 0.1
with torch.no_grad():
    model.weight -= lr * model.weight.grad
print(f"\n模型權重（更新後）：")
print(model.weight.data.round(decimals=3))

## 三、樸素數據並行

今設有 **四器（GPU）**。非以一器處理全部四樣本，
而令各器各處一樣本。每器：
1. 持模型之**完整副本**（初始權重一致）
2. 處理**己之數據分片**
3. 算得**己之梯度**

然後**同步**，使各器皆得相同之平均梯度。

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(14, 3.5))
gpu_colors = ["#4FC3F7", "#81C784", "#FFB74D", "#E57373"]

ax = axes[0]
ax.set_xlim(0, 2); ax.set_ylim(0, 5); ax.axis("off")
ax.set_title("全量批次", fontsize=10, fontweight="bold")
for r in range(4):
    color = gpu_colors[r]
    ax.add_patch(patches.Rectangle((0.2, 3.8 - r * 0.9), 1.6, 0.7,
        facecolor=color, edgecolor="#333", lw=0.8, alpha=0.6))
    ax.text(1.0, 4.15 - r * 0.9, f"樣本 {r}", ha="center", va="center", fontsize=9)

ax = axes[1]
ax.set_xlim(0, 1); ax.set_ylim(0, 5); ax.axis("off")
ax.annotate("分割", xy=(0.7, 2.5), xytext=(0.2, 2.5),
    arrowprops=dict(arrowstyle="->,head_width=0.15", color="#555", lw=2),
    fontsize=11, va="center", color="#555")

for g in range(3):
    ax = axes[g + 2]
    ax.set_xlim(0, 2); ax.set_ylim(0, 5); ax.axis("off")
    ax.set_title(f"GPU {g}", fontsize=10, fontweight="bold", color=gpu_colors[g])
    ax.add_patch(patches.Rectangle((0.2, 3.2), 1.6, 0.7,
        facecolor=gpu_colors[g], edgecolor="#333", lw=0.8, alpha=0.6))
    ax.text(1.0, 3.55, f"樣本 {g}", ha="center", va="center", fontsize=9)
    ax.add_patch(patches.FancyBboxPatch((0.1, 1.2), 1.8, 1.3, boxstyle="round,pad=0.04",
        facecolor="#e3f2fd", edgecolor="#1565c0", lw=1.2))
    ax.text(1.0, 1.85, "模型副本", ha="center", va="center", fontsize=9, fontweight="bold")
    if g == 2:
        ax.text(1.0, 0.7, "... + GPU 3", ha="center", fontsize=9, color="#999")

plt.suptitle("其一 — 分數據，複模型於各器", fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

### 以實張量模擬數據並行

以 `simulate_allreduce` 於 CPU 上模擬器間通信。

In [ ]:
torch.manual_seed(42)
num_gpus = 4

# 相同初始權重（複製於各"GPU"）
W_init = torch.randn(3, 2)
print("初始權重（各器皆同）：")
print(W_init.round(decimals=3))
print()

# 四樣本分於四器
all_X = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [0.5, 0.5]])
all_y = torch.tensor([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [1.0, 1.0, 0.0], [0.5, 0.5, 0.0]])

local_grads = []
for i in range(num_gpus):
    W = W_init.clone().requires_grad_(True)
    x_i = all_X[i:i+1]
    y_i = all_y[i:i+1]
    pred = x_i @ W.T
    loss = ((pred - y_i) ** 2).mean()
    loss.backward()
    local_grads.append(W.grad.clone())
    print(f"GPU {i}: 樣本={all_X[i].tolist()}, 損失={loss.item():.4f}, 梯度範數={W.grad.norm():.4f}")

print("\n--- 各器本地梯度（數據不同，故梯度亦異）---")
for i, g in enumerate(local_grads):
    print(f"  GPU {i}: {g[0].tolist()}")

### 其二 — 同步梯度（AllReduce）

要術在此：**AllReduce** 匯總各器梯度，使每器皆得同一總和。再除以 N 得均值。

> **AllReduce** 者，集體通信之操作也——每參與者既*發*己之數據，又*收*匯總之結果。
> 猶群中眾人各述其答，末了人人皆見其和。

In [ ]:
# AllReduce：各器皆得所有梯度之和
synced = simulate_allreduce(local_grads)
avg_grads = [g / num_gpus for g in synced]

print("AllReduce + 取均值後：")
for i in range(num_gpus):
    print(f"  GPU {i}: {avg_grads[i][0].round(decimals=4).tolist()}")

print(f"\n皆同一否？ {all(torch.allclose(avg_grads[0], avg_grads[i]) for i in range(1, num_gpus))}")

# 與單器處理全部數據之結果對比
W_full = W_init.clone().requires_grad_(True)
pred_full = all_X @ W_full.T
loss_full = ((pred_full - all_y) ** 2).mean()
loss_full.backward()
print(f"\n單器梯度（全部數據）：  {W_full.grad[0].round(decimals=4).tolist()}")
print(f"DP 平均梯度：             {avg_grads[0][0].round(decimals=4).tolist()}")
print(f"完全一致：{torch.allclose(avg_grads[0], W_full.grad)}")

In [ ]:
info_box(
    "四器各處四分之一數據，取平均梯度，與一器處理全部數據之梯度<b>數學上完全一致</b>。"
    "數據並行所得結果全同——惟更速耳！",
    title="要旨"
)

### AllReduce 之形

器間通信如圖。於**環形（Ring）**拓撲中，數據沿環而流，無單點之瓶頸：

In [ ]:
fig, ax = draw_comm_pattern("allreduce", num_gpus=4)
plt.show()

## 四、DistributedDataParallel（DDP）— 巧術版

樸素 DP 可用，然 PyTorch 之 `DistributedDataParallel` 以二巧使之**更速**：

### 巧一：梯度分桶（Gradient Bucketing）

DDP 不待所有梯度算畢方同步，而將梯度分為**桶**（約 25 MB 一桶），
每桶就緒即刻同步。

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_xlim(0, 12); ax.set_ylim(0, 5.5)
ax.axis("off")
ax.set_title("DDP：計算與通信重疊", fontsize=13, fontweight="bold", pad=10)

y_comp = 4.0; y_comm = 2.5
bar_h = 0.8

# 反向計算
layers = ["第四層\n(末層)", "第三層", "第二層", "第一層\n(首層)"]
layer_colors = ["#ef9a9a", "#f48fb1", "#ce93d8", "#b39ddb"]
for i, (label, color) in enumerate(zip(layers, layer_colors)):
    x = 1 + i * 2.2
    ax.add_patch(patches.FancyBboxPatch((x, y_comp), 2.0, bar_h, boxstyle="round,pad=0.03",
        facecolor=color, edgecolor="#333", lw=1))
    ax.text(x + 1.0, y_comp + bar_h/2, f"反向\n{label}", ha="center", va="center", fontsize=8, fontweight="bold")

# 通信
comm_colors = ["#80cbc4", "#80deea", "#81d4fa", "#90caf9"]
for i in range(4):
    x = 2.2 + i * 2.2
    w = 2.5
    if x + w > 11: w = 11 - x
    ax.add_patch(patches.FancyBboxPatch((x, y_comm), w, bar_h, boxstyle="round,pad=0.03",
        facecolor=comm_colors[i], edgecolor="#333", lw=1))
    ax.text(x + w/2, y_comm + bar_h/2, f"AllReduce\n桶 {i}", ha="center", va="center",
        fontsize=8, fontweight="bold")

ax.text(0.5, y_comp + bar_h/2, "計算:", ha="right", va="center", fontsize=10, fontweight="bold")
ax.text(0.5, y_comm + bar_h/2, "通信:", ha="right", va="center", fontsize=10, fontweight="bold")

ax.annotate("重疊!\n(省時)", xy=(5, 3.3), xytext=(9.5, 4.0),
    arrowprops=dict(arrowstyle="->,head_width=0.1", color="#d32f2f", lw=1.5),
    fontsize=11, fontweight="bold", color="#d32f2f", ha="center")

ax.annotate("", xy=(10.5, 0.5), xytext=(1, 0.5),
    arrowprops=dict(arrowstyle="->,head_width=0.12", color="#333", lw=1.5))
ax.text(5.5, 0.2, "時間 →", ha="center", fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
info_box(
    "<b>梯度分桶</b> — DDP 非逐梯度同步（訊息太碎），亦非全算畢再同步（等候太久）。"
    "而是分梯度為約 25 MB 之桶，每桶就緒即同步。如此通信與反向傳播<i>重疊</i>，"
    "大半同步之耗皆隱矣。",
    title="分桶何以重要"
)

## 五、存儲之患 — 何以需 ZeRO

標準 DP 將**一切**複於每器。觀其浪費幾何：

In [ ]:
# 混合精度 + Adam 下的存儲分析
def memory_breakdown(num_params_B, num_gpus, zero_stage=0):
    """計算每器存儲占用（GB），混合精度 + Adam。"""
    P = num_params_B * 1e9
    if zero_stage == 0:
        params_gb = P * 2 / 1e9
        grads_gb = P * 2 / 1e9
        opt_gb = P * 12 / 1e9
    elif zero_stage == 1:
        params_gb = P * 2 / 1e9
        grads_gb = P * 2 / 1e9
        opt_gb = P * 12 / 1e9 / num_gpus
    elif zero_stage == 2:
        params_gb = P * 2 / 1e9
        grads_gb = P * 2 / 1e9 / num_gpus
        opt_gb = P * 12 / 1e9 / num_gpus
    else:
        params_gb = P * 2 / 1e9 / num_gpus
        grads_gb = P * 2 / 1e9 / num_gpus
        opt_gb = P * 12 / 1e9 / num_gpus
    return params_gb, grads_gb, opt_gb

model_size = 7
n_gpus = 8

fig, axes = plt.subplots(1, 4, figsize=(14, 5))
stages = ["標準 DP\n(無 ZeRO)", "ZeRO 一階\n(分片優化器)", "ZeRO 二階\n(+ 分片梯度)", "ZeRO 三階\n(全部分片)"]
bar_colors = ["#42a5f5", "#66bb6a", "#ffa726"]
labels = ["參數", "梯度", "優化器狀態"]

for idx, (ax, title) in enumerate(zip(axes, stages)):
    p, g, o = memory_breakdown(model_size, n_gpus, zero_stage=idx)
    heights = [p, g, o]
    bottom = 0
    for h, c, lab in zip(heights, bar_colors, labels):
        ax.bar(0, h, bottom=bottom, color=c, edgecolor="white", width=0.5)
        if h > 1.5:
            ax.text(0, bottom + h/2, f"{lab}\n{h:.1f} GB", ha="center", va="center", fontsize=8, fontweight="bold")
        elif h > 0.3:
            ax.text(0, bottom + h/2, f"{h:.1f}", ha="center", va="center", fontsize=7)
        bottom += h
    total = sum(heights)
    ax.text(0, bottom + 1, f"{total:.1f} GB", ha="center", fontsize=11, fontweight="bold", color="#333")
    ax.set_title(title, fontsize=9, fontweight="bold")
    ax.set_xlim(-0.8, 0.8); ax.set_ylim(0, 120); ax.set_xticks([])
    if idx == 0: ax.set_ylabel("每器存儲 (GB)", fontsize=10)

for ax in axes:
    ax.axhline(y=80, color="red", linestyle="--", alpha=0.5, lw=1)
    ax.text(0.35, 82, "A100 80GB", fontsize=7, color="red", alpha=0.7)

fig.suptitle(f"每器存儲 — {model_size}B 參數模型, {n_gpus} 器（混合精度 + Adam）",
    fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

rows = []
for stage in range(4):
    p, g, o = memory_breakdown(model_size, n_gpus, zero_stage=stage)
    total = p + g + o
    fits = "可" if total < 80 else "否"
    rows.append([f"階段 {stage}" if stage > 0 else "無 (DDP)", f"{p:.1f}", f"{g:.1f}", f"{o:.1f}", f"{total:.1f}", fits])
comparison_table(
    ["ZeRO 階段", "參數 (GB)", "梯度 (GB)", "優化器 (GB)", "總計 (GB)", "可入 A100 80GB？"],
    rows,
    title=f"存儲分析：{model_size}B 模型，{n_gpus} 器"
)

### ZeRO 分片之法

要旨：**何以每器皆存一份全同之物？**
令各器僅持不時刻需之部分的 1/N 即可。

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
titles = ["標準 DP", "ZeRO-1: 分片優化器", "ZeRO-2: + 分片梯度", "ZeRO-3: 全部分片"]

for ax_idx, (ax, title) in enumerate(zip(axes, titles)):
    ax.set_xlim(0, 5); ax.set_ylim(0, 5); ax.axis("off")
    ax.set_title(title, fontsize=9, fontweight="bold")

    for gpu in range(4):
        x = 0.2 + gpu * 1.2
        ax.text(x + 0.45, 4.7, f"GPU{gpu}", ha="center", fontsize=7, fontweight="bold")

        if ax_idx < 3:
            ax.add_patch(patches.Rectangle((x, 3.5), 0.9, 0.9, facecolor="#42a5f5", edgecolor="white", lw=0.5))
            ax.text(x + 0.45, 3.95, "P", ha="center", va="center", fontsize=8, color="white", fontweight="bold")
        else:
            ax.add_patch(patches.Rectangle((x, 3.5), 0.9, 0.9, facecolor="#bbdefb", edgecolor="white", lw=0.5))
            ax.text(x + 0.45, 3.95, "P/4", ha="center", va="center", fontsize=7, color="#666", fontweight="bold")

        if ax_idx < 2:
            ax.add_patch(patches.Rectangle((x, 2.4), 0.9, 0.9, facecolor="#66bb6a", edgecolor="white", lw=0.5))
            ax.text(x + 0.45, 2.85, "G", ha="center", va="center", fontsize=8, color="white", fontweight="bold")
        else:
            ax.add_patch(patches.Rectangle((x, 2.4), 0.9, 0.9, facecolor="#c8e6c9", edgecolor="white", lw=0.5))
            ax.text(x + 0.45, 2.85, "G/4", ha="center", va="center", fontsize=7, color="#555", fontweight="bold")

        if ax_idx == 0:
            ax.add_patch(patches.Rectangle((x, 0.2), 0.9, 2.0, facecolor="#ffa726", edgecolor="white", lw=0.5))
            ax.text(x + 0.45, 1.2, "Opt", ha="center", va="center", fontsize=8, color="white", fontweight="bold")
        else:
            ax.add_patch(patches.Rectangle((x, 0.2), 0.9, 2.0, facecolor="#ffe0b2", edgecolor="white", lw=0.5))
            ax.text(x + 0.45, 1.2, "Opt/4", ha="center", va="center", fontsize=7, color="#555", fontweight="bold")

fig.suptitle("各器所存之物（深色=完整副本，淺色=1/N 分片）", fontsize=11, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

## 六、FSDP — PyTorch 原生 ZeRO 三階

**全分片數據並行（FSDP）**乃 PyTorch 內建之 ZeRO Stage 3 實現也。

FSDP 中參數之生命週期：

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.set_xlim(0, 13); ax.set_ylim(0, 4.5)
ax.axis("off")
ax.set_title("FSDP：一訓練步中參數之生命週期", fontsize=13, fontweight="bold", pad=10)

steps = [
    (0.3,  "靜止：\n僅存\n1/N 分片", "#e3f2fd", "#1565c0"),
    (2.5,  "AllGather：\n重建完整\n參數", "#c8e6c9", "#2e7d32"),
    (4.7,  "前向\n傳播", "#fff9c4", "#f57f17"),
    (6.9,  "釋非\n本地分片", "#ffcdd2", "#c62828"),
    (8.5,  "AllGather\n(反向傳播\n復需之)", "#c8e6c9", "#2e7d32"),
    (10.5, "反向 +\nReduceScatter", "#e1bee7", "#6a1b9a"),
]

for i, (x, label, fc, ec) in enumerate(steps):
    w = 1.8
    ax.add_patch(patches.FancyBboxPatch((x, 1.2), w, 2.0, boxstyle="round,pad=0.05",
        facecolor=fc, edgecolor=ec, lw=1.5))
    ax.text(x + w/2, 2.2, label, ha="center", va="center", fontsize=9, fontweight="bold")
    ax.text(x + w/2, 0.7, f"步 {i+1}", ha="center", fontsize=8, color="#888")
    if i < len(steps) - 1:
        nx = steps[i+1][0]
        ax.annotate("", xy=(nx - 0.05, 2.2), xytext=(x + w + 0.05, 2.2),
            arrowprops=dict(arrowstyle="->,head_width=0.08", color="#555", lw=1.2))

plt.tight_layout()
plt.show()

In [ ]:
info_box(
    "<b>AllGather</b> — 各器發己之分片予所有他器，眾器皆得重建完整張量。"
    "猶眾人各出拼圖之一片，人人皆見全圖。<br><br>"
    "<b>ReduceScatter</b> — 其反也：先匯眾器之數據（求和），再予各器僅其 1/N 之結果。"
    "歸約與分發合為一步。",
    title="新術語：AllGather 與 ReduceScatter"
)

### 通信開銷：DDP 與 FSDP

FSDP 以**更多通信**換**更少存儲**：

In [ ]:
comparison_table(
    ["", "DDP", "FSDP (ZeRO-3)"],
    [
        ["每器存儲", "完整模型 + 梯度 + 優化器", "一切之 1/N"],
        ["每步通信量", "2 × 模型大小<br>(1 AllReduce = ReduceScatter + AllGather)",
         "3 × 模型大小<br>(AllGather×2 + ReduceScatter)"],
        ["相對 DDP", "1×", "1.5×"],
        ["適用", "模型可入單器存儲", "模型不能入單器"],
    ],
    title="DDP 與 FSDP 之權衡"
)

## 七、數據並行於大語言模型

實踐中，DP 乃**一切**大模型訓練之基石：

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_xlim(0, 12); ax.set_ylim(0, 5.5)
ax.axis("off")
ax.set_title("何等模型用何等並行？", fontsize=13, fontweight="bold", pad=10)

regions = [
    (0.5, 0.5, 3.5, 4.0, "#e3f2fd", "小模型\n(< 10B)",
     "純 DDP\n\nBERT, GPT-2,\nLLaMA-7B", "#1565c0"),
    (4.3, 0.5, 3.5, 4.0, "#fff3e0", "中等模型\n(10B – 100B)",
     "FSDP 或\nDP + TP\n\nLLaMA-65B,\nGPT-NeoX-20B", "#e65100"),
    (8.1, 0.5, 3.5, 4.0, "#fce4ec", "大模型\n(> 100B)",
     "DP + TP + PP\n(三維並行)\n\nGPT-3-175B,\nMegatron-530B", "#c62828"),
]
for x, y, w, h, fc, title, desc, ec in regions:
    ax.add_patch(patches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.08",
        facecolor=fc, edgecolor=ec, lw=2))
    ax.text(x + w/2, y + h - 0.4, title, ha="center", va="center", fontsize=11, fontweight="bold", color=ec)
    ax.text(x + w/2, y + h/2 - 0.3, desc, ha="center", va="center", fontsize=9, color="#333")

ax.annotate("", xy=(11.2, 4.8), xytext=(0.5, 4.8),
    arrowprops=dict(arrowstyle="->,head_width=0.12", color="#2e7d32", lw=2.5))
ax.text(6, 5.15, "← 數據並行恆為最外層 →",
    ha="center", fontsize=11, fontweight="bold", color="#2e7d32")

plt.tight_layout()
plt.show()

In [ ]:
info_box(
    "於多維並行（如 TP × PP × DP）中，數據並行恆為<b>最外層</b>，蓋：<br>"
    "1. 每步僅需通信一次（梯度同步）<br>"
    "2. 線性擴展——器倍則吞吐量近倍<br>"
    "3. 無流水線氣泡或激活值存儲之額外開銷",
    title="何以 DP 恆居外層"
)

### 擴展效率

器數增則通信開銷亦增。可視化之：

In [ ]:
model_sizes_B = [1, 7, 13]
gpu_counts = [1, 2, 4, 8, 16, 32, 64]
bandwidth_gbps = 300

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

for model_B in model_sizes_B:
    compute_ms = model_B * 15
    effs = []
    for n in gpu_counts:
        if n == 1:
            effs.append(1.0)
            continue
        data_gb = model_B * 1e9 * 4 / 1e9
        comm_ms = 2 * (n-1)/n * data_gb / bandwidth_gbps * 1000
        eff = compute_ms / (compute_ms + comm_ms)
        effs.append(eff)
    ax1.plot(gpu_counts, effs, "o-", lw=2, label=f"{model_B}B 參數")

ax1.set_xlabel("GPU 數", fontsize=10)
ax1.set_ylabel("擴展效率", fontsize=10)
ax1.set_title("DP 效率 vs GPU 數量", fontweight="bold")
ax1.legend(); ax1.grid(True, alpha=0.3); ax1.set_ylim(0, 1.05)

dp_sizes = [1, 2, 4, 8, 16, 32, 64]
local_batch = 32
ax2.bar(range(len(dp_sizes)), [local_batch * n for n in dp_sizes], color="#42a5f5")
ax2.set_xticks(range(len(dp_sizes))); ax2.set_xticklabels(dp_sizes)
ax2.set_xlabel("DP GPU 數", fontsize=10)
ax2.set_ylabel("有效批次大小", fontsize=10)
ax2.set_title("批次大小隨 DP 線性增長", fontweight="bold")
ax2.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

## 八、實戰：真實 DDP 訓練腳本

下為一**完整可運行**之 DDP 訓練腳本。存之，於多卡機器上運行：

```bash
# 於多卡 GPU 機器:
torchrun --nproc_per_node=4 ddp_train.py
```

In [ ]:
gpu_required_banner()

In [ ]:
# [GPU-REQUIRED]
# 完整 DDP 訓練腳本 — 存為 ddp_train.py

DDP_SCRIPT = '''\
import os
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, TensorDataset, DistributedSampler

def main():
    # 1. 初始化分佈式 — GPU 使用 NCCL 後端
    dist.init_process_group(backend="nccl")
    rank = dist.get_rank()          # 當前 GPU 之 ID (0, 1, 2, ...)
    world_size = dist.get_world_size()  # GPU 總數
    device = torch.device(f"cuda:{rank}")
    torch.cuda.set_device(device)

    # 2. 建模型，置於當前 GPU
    model = nn.Sequential(
        nn.Linear(128, 256), nn.ReLU(), nn.Linear(256, 10)
    ).to(device)

    # 3. 以 DDP 包裹 — 梯度同步自動完成
    model = DDP(model, device_ids=[rank])

    # 4. 數據集 + DistributedSampler（自動於器間分割數據）
    X = torch.randn(1000, 128)
    y = torch.randint(0, 10, (1000,))
    sampler = DistributedSampler(TensorDataset(X, y), num_replicas=world_size, rank=rank)
    loader = DataLoader(TensorDataset(X, y), batch_size=32, sampler=sampler)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    # 5. 訓練
    for epoch in range(5):
        sampler.set_epoch(epoch)   # 每 epoch 不同 shuffle
        total_loss = 0.0
        for batch_X, batch_y in loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(batch_X), batch_y)
            loss.backward()        # DDP 於此自動同步梯度！
            optimizer.step()
            total_loss += loss.item()
        if rank == 0:
            print(f"Epoch {epoch+1}: loss = {total_loss/len(loader):.4f}")

    dist.destroy_process_group()

if __name__ == "__main__":
    main()
'''

print(DDP_SCRIPT)
print("# 運行：torchrun --nproc_per_node=4 ddp_train.py")

## 九、Megatron-LM 參考

於 Megatron 之三維並行框架中，DP 與 TP、PP 合用。關鍵實現：

In [ ]:
code_reference(
    code="""# Megatron-LM 將諸器組為進程組。
# 數據並行組 = 同 TP rank + 同 PP stage 之器，
# 然處不同之數據分片。
#
# 例：16 器, TP=4, PP=1, DP=4
# TP 組: [0,1,2,3], [4,5,6,7], [8,9,10,11], [12,13,14,15]
# DP 組: [0,4,8,12], [1,5,9,13], [2,6,10,14], [3,7,11,15]
#
# 各 DP 組獨立行 AllReduce。
# megatron/core/parallel_state.py""",
    source="Megatron-LM",
    filepath="megatron/core/parallel_state.py"
)

In [ ]:
code_reference(
    code="""# Megatron 用己之 DDP 包裹器（非 PyTorch 默認者）
# 蓋模型已為 TP/PP 所切分。
#
# megatron/core/distributed/distributed_data_parallel.py
class DistributedDataParallel(MegatronModule):
    '''於數據並行進程組內行梯度 AllReduce。

    與 torch DDP 之要異：
    - 僅於 DP 組內 AllReduce（非所有器）
    - 適用於已為 TP、PP 切分之模型
    - 支持跨微批次之梯度累積
    '''""",
    source="Megatron-LM",
    filepath="megatron/core/distributed/distributed_data_parallel.py"
)

## 十、總結

此篇所述：

In [ ]:
comparison_table(
    ["術", "要旨", "存儲", "通信量", "適用"],
    [
        ["樸素 DP", "複模型，分數據，均梯度",
         "每器完整副本", "樸素聚合 → 瓶頸", "僅供理解"],
        ["DDP", "AllReduce + 分桶 + 重疊",
         "每器完整副本", "2× 模型 (高效)", "模型可入單器"],
        ["ZeRO-1", "分片優化器狀態",
         "優化器存儲減約四倍", "同 DDP", "優化器為瓶頸"],
        ["ZeRO-2", "ZeRO-1 + 分片梯度",
         "更少", "同 DDP", "需更多節省"],
        ["ZeRO-3 / FSDP", "全部分片",
         "一切之 1/N", "3× 模型 (1.5× DDP)", "模型不入單器"],
    ],
    title="數據並行家族 — 總覽"
)

### 要式

| 式 | 義 |
|---|---|
| $g_{\text{avg}} = \frac{1}{N} \sum_{i=1}^{N} g_i$ | N 器之梯度取均 |
| $B_{\text{eff}} = b_{\text{local}} \times N_{\text{DP}} \times \text{grad\_accum}$ | 有效批次大小 |
| DDP 通信: $2 \times |\theta|$ | AllReduce = ReduceScatter + AllGather |
| FSDP 通信: $3 \times |\theta|$ | 2× AllGather + 1× ReduceScatter |

### 後篇

**[二 — 張量並行](../02-tensor-parallelism/)**：切分單層於多器（與 DP 互補）。

### 延伸

- [ZeRO: Memory Optimizations Toward Training Trillion Parameter Models](https://arxiv.org/abs/1910.02054)
- [PyTorch DDP Tutorial](https://pytorch.org/tutorials/beginner/dist_overview.html)
- [PyTorch FSDP Tutorial](https://pytorch.org/tutorials/intermediate/FSDP_tutorial.html)
- [Efficient Large-Scale Language Model Training on GPU Clusters](https://arxiv.org/abs/2104.04473)